# Clustering: Complete Notebook
> **Course:** Machine Learning / Business Intelligence
> **Goal:** Discover latent structure in unlabeled data using K-Means, Hierarchical Clustering and DBSCAN

---
## What will you learn?
1. Identify the geometry of different kinds of data
2. Preprocess data: encoding and scaling
3. Apply **K-Means**, **Hierarchical Clustering** and **DBSCAN**
4. Choose the optimal number of clusters (Elbow, Silhouette)
5. Evaluate quality with internal metrics
6. Segment real bank customers

---
### Why clustering?
Clustering is **unsupervised learning**: there are no labels, the algorithm discovers structure on its own.

| Algorithm | Shape assumption | Needs k | Handles noise |
|---|---|---|---|
| K-Means | Convex spheres | Yes | No |
| Hierarchical | Any (bottom-up) | No | No |
| DBSCAN | Any (density) | No | Yes |
| Mean Shift | Any (kernel) | No | Partially |

## 1. Installation and Setup

In [ ]:
!pip install scikit-learn scipy matplotlib seaborn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift, estimate_bandwidth
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist

plt.rcParams.update({
    'figure.facecolor': '#18181b', 'axes.facecolor': '#27272a',
    'axes.edgecolor': '#3f3f46', 'text.color': '#e4e4e7',
    'axes.labelcolor': '#a1a1aa', 'xtick.color': '#71717a',
    'ytick.color': '#71717a', 'grid.color': '#3f3f46',
    'grid.alpha': 0.4, 'axes.titlesize': 12,
})

PALETTE  = ['#10b981','#3b82f6','#f59e0b','#8b5cf6','#ef4444','#ec4899']
NOISE_C  = '#52525b'
np.random.seed(42)
print('Ready')

---
## 2. Datasets: Clustering Geometries

Not every algorithm works well on every kind of data. The **geometry** of the clusters matters.

| Dataset | Geometry | Ideal algorithm |
|---|---|---|
| **Blobs** | Well-separated spheres | K-Means |
| **Moons** | Interleaved half-moons | DBSCAN |
| **Circles** | Concentric circles | DBSCAN |
| **Aniso** | Rotated ellipses | K-Means fails, DBSCAN works |
| **Varied** | Clusters with different variance | Hierarchical |
| **Noise** | Uniform noise | DBSCAN (identifies noise) |

In [ ]:
N = 300

blobs,   y_blobs   = make_blobs(n_samples=N, centers=3, cluster_std=0.8, random_state=42)
moons,   y_moons   = make_moons(n_samples=N, noise=0.08, random_state=42)
circles, y_circles = make_circles(n_samples=N, factor=0.5, noise=0.07, random_state=42)

X_aniso = np.dot(make_blobs(n_samples=N, random_state=170)[0],
                 [[0.60, -0.60], [-0.40, 0.80]])
y_aniso = make_blobs(n_samples=N, random_state=170)[1]

X_varied, y_varied = make_blobs(n_samples=N, cluster_std=[0.5, 1.5, 0.3], random_state=42)

X_noise = np.random.uniform(-3, 3, size=(N, 2))
y_noise = np.zeros(N, dtype=int)

DATASETS = {
    'Blobs':   (blobs, y_blobs),
    'Moons':   (moons, y_moons),
    'Circles': (circles, y_circles),
    'Aniso':   (X_aniso, y_aniso),
    'Varied':  (X_varied, y_varied),
    'Noise':   (X_noise, y_noise),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Clustering Datasets: Different Geometries', fontsize=14)

for ax, (name, (X, y)) in zip(axes.flat, DATASETS.items()):
    colors = [PALETTE[int(c) % len(PALETTE)] for c in y]
    ax.scatter(X[:,0], X[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 3. Preprocessing: Encoding and Scaling

Distance-based algorithms (K-Means, Hierarchical) are **very sensitive to scale**.

### Why scale?
If `Age` ranges from 20 to 70 and `Income` from 20,000 to 200,000, the Euclidean distance will be **dominated by income** even though both variables are equally important.

| Method | Formula | Result | When to use |
|---|---|---|---|
| **Min-Max** | `(x - min) / (max - min)` | [0, 1] | Known distribution, no extreme outliers |
| **Z-Score** | `(x - μ) / σ` | mean=0, std=1 | Approximately normal distribution |

In [ ]:
# Demonstration: effect of scaling on K-Means
np.random.seed(42)
n = 200
age    = np.random.normal(40, 10, n)           # 20 to 70 years old
income = np.random.normal(60000, 20000, n)     # 20k to 100k
X_demo = np.column_stack([age, income])

scaler_mm = MinMaxScaler()
scaler_zs = StandardScaler()
X_mm = scaler_mm.fit_transform(X_demo)
X_zs = scaler_zs.fit_transform(X_demo)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Effect of Scaling on the Data')

labels_raw = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_demo)
labels_mm  = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_mm)
labels_zs  = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_zs)

for ax, X, labels, title in [
    (axes[0], X_demo, labels_raw, 'Unscaled\n(Income dominates the distance)'),
    (axes[1], X_mm,   labels_mm,  'Min-Max Scaling\n(Both in [0,1])'),
    (axes[2], X_zs,   labels_zs,  'Z-Score Standardization\n(Both on a σ scale)'),
]:
    colors = [PALETTE[l] for l in labels]
    ax.scatter(X[:,0], X[:,1], c=colors, s=15, alpha=0.8, edgecolors='none')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Age (or scaled)'); ax.set_ylabel('Income (or scaled)')
    sil = silhouette_score(X, labels)
    ax.text(0.02, 0.96, f'Silhouette={sil:.3f}', transform=ax.transAxes,
            fontsize=8, color='#fbbf24', va='top')

plt.tight_layout()
plt.show()
print('Note: without scaling, the clusters are separated by income alone (vertical axis).')

In [ ]:
# Encoding categorical variables
df_example = pd.DataFrame({
    'ciudad':         ['Santiago', 'Valparaíso', 'Concepción', 'Santiago', 'Concepción'],
    'estado_civil':   ['Soltero', 'Casado', 'Divorciado', 'Casado', 'Soltero'],
    'ingreso':        [45000, 60000, 38000, 75000, 52000],
})

print('Original data:')
print(df_example)
print()

# One-Hot Encoding
df_ohe = pd.get_dummies(df_example, columns=['ciudad','estado_civil'], dtype=int)
print('One-Hot Encoding (one column per category):')
print(df_ohe)
print()

# Label Encoding (only when there is a natural order)
le = LabelEncoder()
df_le = df_example.copy()
df_le['ciudad_code']       = le.fit_transform(df_example['ciudad'])
df_le['estado_civil_code'] = le.fit_transform(df_example['estado_civil'])
print('Label Encoding (numeric code; careful: it imposes an artificial order):')
print(df_le[['ciudad','ciudad_code','estado_civil','estado_civil_code','ingreso']])

---
## 4. K-Means

### Algorithm (Lloyd's Algorithm)
1. Initialize `k` centroids at random (or with K-Means++)
2. **Assign** each point to the nearest centroid
3. **Update** the centroids by computing the mean of the assigned points
4. Repeat until convergence

**Objective function:** minimize the sum of squared distances (WCSS/Inertia)
$$\text{WCSS} = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

In [ ]:
# Elbow method: find the optimal k
X_scaled = StandardScaler().fit_transform(blobs)

k_range = range(1, 11)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_ for k in k_range]
sil_scores = [silhouette_score(X_scaled, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled))
              if k > 1 else 0 for k in k_range]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Choosing the Optimal k for K-Means')

axes[0].plot(list(k_range), wcss, marker='o', color='#10b981', linewidth=2)
axes[0].fill_between(list(k_range), wcss, alpha=0.15, color='#10b981')
axes[0].axvline(3, color='#f59e0b', linewidth=1.5, linestyle='--', label='elbow at k=3')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(list(k_range)[1:], sil_scores[1:], marker='s', color='#3b82f6', linewidth=2)
axes[1].fill_between(list(k_range)[1:], sil_scores[1:], alpha=0.15, color='#3b82f6')
axes[1].axvline(3, color='#f59e0b', linewidth=1.5, linestyle='--', label='maximum at k=3')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k (higher = better)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# K-Means on the 6 datasets
K_VALUES = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3, 'Varied': 3, 'Noise': 3}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('K-Means on Different Geometries')

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    k  = K_VALUES[name]
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(Xs)
    sil = silhouette_score(Xs, labels) if len(set(labels)) > 1 else 0
    colors = [PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.75, edgecolors='none')
    ax.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
               c='white', s=120, marker='*', zorder=5, edgecolors='#f59e0b', linewidths=1)
    ax.set_title(f'{name}  (k={k}, sil={sil:.2f})')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()
print('K-Means fails on Moons and Circles (it assumes spherical clusters).')
print('Note the low silhouette score in those cases.')

---
## 5. Hierarchical Clustering

### Algorithm (Agglomerative, Bottom-Up)
1. Each point starts as its own cluster
2. Merge the two closest clusters
3. Repeat until there is a single cluster
4. **Cut the dendrogram** at the height that gives the desired number of clusters

The result is visualized as a **dendrogram**, a hierarchical tree of merges.

In [ ]:
# Compare linkage types
X_hier = StandardScaler().fit_transform(blobs[:100])  # subsample for visualization

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Dendrograms: Comparison of Linkage Methods')

for ax, method, color in zip(axes, ['ward','complete','average','single'],
                              ['#10b981','#3b82f6','#f59e0b','#8b5cf6']):
    Z = linkage(X_hier, method=method)
    dendrogram(Z, ax=ax, color_threshold=0,
               above_threshold_color=color,
               leaf_font_size=0, no_labels=True)
    ax.set_title(f'Linkage: {method.capitalize()}')
    ax.set_ylabel('Distance')
    ax.axhline(y=2.0, color='#ef4444', linewidth=1, linestyle='--', label='cut')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Hierarchical clustering on all datasets
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Hierarchical Clustering (Ward Linkage) on Different Geometries')

N_CLUSTERS = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3, 'Varied': 3, 'Noise': 4}

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    n  = N_CLUSTERS[name]
    labels = AgglomerativeClustering(n_clusters=n, linkage='ward').fit_predict(Xs)
    sil = silhouette_score(Xs, labels)
    colors = [PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.75, edgecolors='none')
    ax.set_title(f'{name}  (n={n}, sil={sil:.2f})')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 6. DBSCAN: Density-Based Spatial Clustering

### Key concepts
- **ε (epsilon):** radius of the neighborhood around each point
- **min_samples:** minimum number of points for a region to be dense
- **Core point:** has >= `min_samples` neighbors within radius ε
- **Border point:** lies within ε of a core point but is not a core point itself
- **Noise:** neither core nor border, labeled as -1

**Advantage:** it finds clusters of **arbitrary shape** and detects **noise automatically**.

In [ ]:
# Effect of epsilon in DBSCAN
X_moons_s = StandardScaler().fit_transform(moons)

eps_values = [0.1, 0.2, 0.3, 0.5]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Effect of epsilon in DBSCAN (Moons, min_samples=5)')

for ax, eps in zip(axes, eps_values):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moons_s)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = np.sum(labels == -1)
    colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(X_moons_s[:,0], X_moons_s[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(f'ε={eps}\n{n_clusters} clusters, {n_noise} noise', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

noise_patch = plt.matplotlib.patches.Patch(color=NOISE_C, label='Noise (-1)')
axes[0].legend(handles=[noise_patch], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN on all datasets
DBSCAN_PARAMS = {
    'Blobs': (0.4, 5), 'Moons': (0.2, 5), 'Circles': (0.2, 5),
    'Aniso': (0.3, 5), 'Varied': (0.3, 5), 'Noise': (0.15, 5),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('DBSCAN on Different Geometries')

for ax, (name, (X, _)) in zip(axes.flat, DATASETS.items()):
    Xs = StandardScaler().fit_transform(X)
    eps, min_s = DBSCAN_PARAMS[name]
    labels = DBSCAN(eps=eps, min_samples=min_s).fit_predict(Xs)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = np.sum(labels == -1)
    sil = silhouette_score(Xs, labels) if len(set(labels[labels != -1])) > 1 else 0
    colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
    ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=12, alpha=0.8, edgecolors='none')
    ax.set_title(f'{name}  ({n_clusters} clusters, {n_noise} noise, sil={sil:.2f})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()
print('DBSCAN handles Moons and Circles correctly and detects noise in Noise.')

---
## 7. Algorithm Comparison

When should you use each one? Let's compare the results side by side.

In [ ]:
# Grid: algorithm x dataset
SELECTED_DATASETS = ['Blobs', 'Moons', 'Circles', 'Aniso']
algorithms = {
    'K-Means':      lambda X: KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X),
    'Hierarchical': lambda X: AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X),
    'DBSCAN':       lambda X: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Algorithm Comparison x Data Geometry', fontsize=13)

for col, dname in enumerate(SELECTED_DATASETS):
    X, _ = DATASETS[dname]
    Xs = StandardScaler().fit_transform(X)
    for row, (aname, afn) in enumerate(algorithms.items()):
        ax = axes[row, col]
        labels = afn(Xs)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        sil = silhouette_score(Xs, labels) if n_clusters > 1 else 0
        colors = [NOISE_C if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]
        ax.scatter(Xs[:,0], Xs[:,1], c=colors, s=8, alpha=0.75, edgecolors='none')
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0: ax.set_title(dname, fontsize=10)
        if col == 0: ax.set_ylabel(aname, fontsize=9)
        ax.text(0.02, 0.95, f'sil={sil:.2f}', transform=ax.transAxes,
                fontsize=7, color='#fbbf24', va='top')

plt.tight_layout()
plt.show()

---
## 8. Evaluation Metrics

### Internal metrics (no ground-truth labels)

| Metric | Conceptual formula | Range | Best value |
|---|---|---|---|
| **Silhouette** | (separation - cohesion) / max | [-1, 1] | Close to 1 |
| **Davies-Bouldin** | avg(max dispersion/separation ratio) | [0, ∞) | Close to 0 |
| **WCSS/Inertia** | Σ squared distances to the centroid | [0, ∞) | Relative minimum (elbow) |

In [ ]:
# Detailed silhouette plot
X_s = StandardScaler().fit_transform(blobs)
K_TEST = [2, 3, 4]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Silhouette Plot for k=2,3,4 (Blobs dataset)')

for ax, k in zip(axes, K_TEST):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_s)
    sil_vals = silhouette_samples(X_s, labels)
    avg_sil = silhouette_score(X_s, labels)
    y_lower = 10
    for c in range(k):
        c_sil = np.sort(sil_vals[labels == c])
        size  = len(c_sil)
        y_upper = y_lower + size
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                          color=PALETTE[c], alpha=0.85)
        y_lower = y_upper + 5
    ax.axvline(avg_sil, color='#ef4444', linewidth=2, linestyle='--')
    ax.set_title(f'k={k}  |  avg silhouette={avg_sil:.3f}')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_ylabel('Cluster')
    ax.grid(axis='x')

plt.tight_layout()
plt.show()
print('k=3 has the highest average silhouette and the most uniform bars -> best clustering.')

In [ ]:
# Comparison table of metrics for all algorithms
print('═' * 75)
print(f"{'Algorithm':15s} {'Dataset':10s} {'Clusters':9s} {'Silhouette':12s} {'Davies-Bouldin':16s}")
print('═' * 75)

alg_fns_table = {
    'K-Means':     lambda X, k: KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X),
    'Hierarchical': lambda X, k: AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
    'DBSCAN':      lambda X, k: DBSCAN(eps=0.3, min_samples=5).fit_predict(X),
}
K_TRUE = {'Blobs': 3, 'Moons': 2, 'Circles': 2, 'Aniso': 3}

for dname in ['Blobs', 'Moons', 'Circles', 'Aniso']:
    X, _ = DATASETS[dname]
    Xs = StandardScaler().fit_transform(X)
    for aname, afn in alg_fns_table.items():
        labels = afn(Xs, K_TRUE[dname])
        n_valid = len(set(labels[labels != -1]))
        if n_valid < 2:
            print(f"{aname:15s} {dname:10s} {n_valid:9d} {'N/A':12s} {'N/A':16s}")
            continue
        mask = labels != -1
        sil = silhouette_score(Xs[mask], labels[mask])
        dbi = davies_bouldin_score(Xs[mask], labels[mask])
        print(f"{aname:15s} {dname:10s} {n_valid:9d} {sil:12.4f} {dbi:16.4f}")

print('═' * 75)

---
## 9. Case Study: Bank Customer Segmentation

We are the analytics team at a bank. We have data on **150 customers** with demographic and financial-behavior variables. We want to identify **segments** in order to personalize products and communications.

In [ ]:
np.random.seed(42)
n_clients = 150

# Latent segments:
# A: Young professionals (25-35, medium income, high spending score, low savings)
# B: Established families (35-50, high income, medium spending score, high savings)
# C: Conservative seniors (55-70, medium-high income, low spending, very high savings)

seg_a = int(n_clients * 0.35)
seg_b = int(n_clients * 0.40)
seg_c = n_clients - seg_a - seg_b

ages    = np.concatenate([np.random.normal(30, 4, seg_a), np.random.normal(43, 5, seg_b), np.random.normal(62, 5, seg_c)])
incomes = np.concatenate([np.random.normal(45000, 8000, seg_a), np.random.normal(80000, 15000, seg_b), np.random.normal(65000, 12000, seg_c)])
spending = np.concatenate([np.random.normal(72, 12, seg_a), np.random.normal(55, 10, seg_b), np.random.normal(30, 8, seg_c)])
savings  = np.concatenate([np.random.normal(15, 5, seg_a), np.random.normal(45, 10, seg_b), np.random.normal(65, 10, seg_c)])

true_labels = np.array([0]*seg_a + [1]*seg_b + [2]*seg_c)
cities  = np.random.choice(['Santiago', 'Valparaíso', 'Concepción', 'Temuco'], n_clients)
marital = np.random.choice(['Soltero', 'Casado', 'Divorciado'], n_clients, p=[0.35, 0.50, 0.15])

df_bank = pd.DataFrame({
    'edad':          ages.clip(18, 80).astype(int),
    'ingreso':       incomes.clip(15000, 200000).astype(int),
    'puntaje_gasto': spending.clip(0, 100).round(1),
    'ahorro_pct':    savings.clip(0, 100).round(1),
    'ciudad':        cities,
    'estado_civil':  marital,
})

print(f'Shape: {df_bank.shape}')
print()
df_bank.head(8)

In [ ]:
# Quick EDA
print(df_bank.describe().round(1))
print()
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
fig.suptitle('Distribution of Numeric Variables: Bank Customers')
for ax, col, color in zip(axes,
    ['edad','ingreso','puntaje_gasto','ahorro_pct'],
    ['#10b981','#3b82f6','#f59e0b','#8b5cf6']):
    ax.hist(df_bank[col], bins=20, color=color, alpha=0.85, edgecolor='#18181b')
    ax.set_title(col); ax.grid(axis='y')
plt.tight_layout(); plt.show()

In [ ]:
# Full preprocessing
df_prep = df_bank.copy()

# One-Hot Encoding for categorical variables
df_prep = pd.get_dummies(df_prep, columns=['ciudad', 'estado_civil'], dtype=float)

# Z-Score on numeric variables
num_cols = ['edad', 'ingreso', 'puntaje_gasto', 'ahorro_pct']
scaler = StandardScaler()
df_prep[num_cols] = scaler.fit_transform(df_prep[num_cols])

print(f'Shape after preprocessing: {df_prep.shape}')
print(f'Columns: {list(df_prep.columns)}')

In [ ]:
# K-Means with k=3
X_bank = df_prep.values
km_bank = KMeans(n_clusters=3, n_init=20, random_state=42)
bank_labels = km_bank.fit_predict(X_bank)

# PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_bank)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bank Customer Segmentation: K-Means (k=3)')

# Plot PCA
colors = [PALETTE[l] for l in bank_labels]
axes[0].scatter(X_pca[:,0], X_pca[:,1], c=colors, s=30, alpha=0.8, edgecolors='none')
axes[0].set_title(f'2D PCA: Silhouette={silhouette_score(X_bank, bank_labels):.3f}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
for i, c in enumerate(['Segment A', 'Segment B', 'Segment C']):
    axes[0].scatter([], [], c=PALETTE[i], label=c, s=50)
axes[0].legend()

# Cluster distribution
unique, counts = np.unique(bank_labels, return_counts=True)
axes[1].bar([f'Seg {["A","B","C"][i]}' for i in unique], counts,
             color=[PALETTE[i] for i in unique], edgecolor='#18181b')
axes[1].set_title('Segment Sizes')
axes[1].set_ylabel('Number of customers')
for x, y in zip(range(3), counts):
    axes[1].text(x, y + 0.5, str(y), ha='center', color='#e4e4e7')
axes[1].grid(axis='y')

plt.tight_layout(); plt.show()

In [ ]:
# Profile of each segment
df_bank['segmento'] = bank_labels
seg_names = {0: 'Seg A', 1: 'Seg B', 2: 'Seg C'}
df_bank['segmento_nombre'] = df_bank['segmento'].map(seg_names)

profile = df_bank.groupby('segmento_nombre')[['edad','ingreso','puntaje_gasto','ahorro_pct']].mean().round(1)
print('AVERAGE PROFILE PER SEGMENT:')
print(profile)
print()

# Profile visualization (radar / parallel coordinates style)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
fig.suptitle('Profile of Each Segment')

for ax, col, color in zip(axes,
    ['edad', 'ingreso', 'puntaje_gasto', 'ahorro_pct'],
    ['#10b981', '#3b82f6', '#f59e0b', '#8b5cf6']):
    means = df_bank.groupby('segmento_nombre')[col].mean()
    stds  = df_bank.groupby('segmento_nombre')[col].std()
    ax.bar(means.index, means.values, color=[PALETTE[i] for i in range(3)],
           yerr=stds.values, capsize=4, edgecolor='#18181b', alpha=0.85)
    ax.set_title(col); ax.grid(axis='y')

plt.tight_layout(); plt.show()

In [ ]:
# Business interpretation
print('SEGMENT INTERPRETATION: EXECUTIVE SUMMARY')
print('═' * 65)
for seg_id, name in seg_names.items():
    grp = df_bank[df_bank['segmento'] == seg_id]
    edad_m  = grp['edad'].mean()
    ing_m   = grp['ingreso'].mean()
    gasto_m = grp['puntaje_gasto'].mean()
    aho_m   = grp['ahorro_pct'].mean()
    n       = len(grp)
    label = (
        'Young Professionals'  if edad_m < 40 else
        'Established Family'   if edad_m < 55 else
        'Conservative Senior'
    )
    print(f"\n  {name}: {label} (n={n})")
    print(f"    Age: {edad_m:.0f}  |  Income: ${ing_m:,.0f}")
    print(f"    Spending: {gasto_m:.0f}/100  |  Savings: {aho_m:.0f}%")
    action = (
        '-> Consumer credit, rewards credit cards, digital checking account' if edad_m < 40 else
        '-> Mortgage, life insurance, children\'s education fund'         if edad_m < 55 else
        '-> Conservative mutual funds, health insurance, estate planning'
    )
    print(f"    Recommended products: {action}")
print()

---
## 10. Exercises

### Exercise 1: The wrong algorithm for the geometry
Apply K-Means with k=2 to the **Circles** dataset. What silhouette score do you get? Compare with DBSCAN. Why does K-Means fail here?

### Exercise 2: Sensitivity to epsilon in DBSCAN
For the **Moons** dataset, try ε = 0.05, 0.15, 0.30, 0.50. Plot the number of clusters and noise points detected for each value. What is the optimal range for ε?

### Exercise 3: The importance of scaling
Apply K-Means to the bank dataset **without** scaling. Compare the silhouette score and the segment profiles with the scaled version. Which variable dominates when you do not scale?

### Exercise 4: Cutting the dendrogram
For the bank dataset, build a dendrogram (Ward linkage). At what height would you cut it to get 3 clusters? And 4? Visualize both solutions.

### Exercise 5: A new domain
Change the case study to an **e-commerce store**: the variables would be `dias_desde_ultima_compra` (days since last purchase), `frecuencia_compras` (purchase frequency), `valor_promedio_pedido` (average order value), the classic RFM analysis. Generate synthetic data and repeat the full analysis.